In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

# Path to Custom Modules:

In [2]:
import os
import sys

In [3]:
path_to_folder = os.getcwd()
path_to_custom_modules = path_to_folder + '\\MODULES'
sys.path.append(path_to_custom_modules)

In [4]:
from stokes.retrivalMTM import *
from custom_plotting import complexToRGB
import pickle
import re #regex for creating string


# Path to the experimental data and SST setup:

In [5]:
path_to_stokes_set_up_library = path_to_folder + '\\SST_setup\\StokesTomagraphySetUp_projections_1770.pkl' # Needed for the Gellman matrices and unique eigenvalues of them, this can be regenerated on demand
path_to_stokes_Intensity_Sweep = path_to_folder + '\\experimental_data\\030423_5_mode_groups_BW_40nm_1300nm_N_118_wav' #Acquiered Experimental data for 5 mode groups 2 pols at 0,5,10,20,30 nm bandwidth sources

In [6]:
with open(path_to_stokes_set_up_library,'rb') as file:
     stokesSetUp = pickle.load(file)
stokesSetUp.keys()

C:\Users\ModeLabQBI\AppData\Local\Temp\ipykernel_58124\2855283433.py:2: DeprecationWarning: Please import `csr_matrix` from the `scipy.sparse` namespace; the `scipy.sparse.csr` namespace is deprecated and will be removed in SciPy 2.0.0.
  stokesSetUp = pickle.load(file)


dict_keys(['Gellman', 'stokeStates', 'stokeWeights', 'modegroup', 'polcount'])

In [7]:
# Discover the folder and list the files:
stokes_intensity_results = sorted(os.listdir(path_to_stokes_Intensity_Sweep), key = len)
for fileIdx, filename in enumerate(stokes_intensity_results):
    print(f'{fileIdx} - {filename}') 

0 - BW_0nm
1 - BW_5nm
2 - BW_10nm
3 - BW_20nm
4 - BW_30nm
5 - BW_40nm


In [8]:
name_keyword_path = r"(?P<date>\w+)_(?P<modegroups>\w+)_mode_groups_BW_(?P<bwTotal>\w+)_(?P<wav>\w+)_N_(?P<sampling>\w+)_wav"
name_keyword_file = r"BW_(?P<bw>\w+)"

# Automatic MTM retrival for all $\Delta\lambda_S$

In [10]:
MFD_retrival = 18.8 #Input mode field diameter for spot to LG conversion during processing of the data. DO NOT CHANGE IT SINCE IT PART OF THE EXPERIMENTAL SETUP

In [ ]:
#This object perform the recovers MTM from experimental data
MTM_retrival_Object = Stokes_Tomography_MTM_retrival(StokesTomographySetUp = stokesSetUp, MFD_retrival = MFD_retrival, forcePSD = False)

In [ ]:
#Iterate across all the data and process it
labels = []
for fileIdx, filename in enumerate(stokes_intensity_results):
    path_to_data = path_to_stokes_Intensity_Sweep + '\\' + filename + '\\'
    fields = re.match(name_keyword_file, filename)
    label = fields.group('bw')
    labels.append(label)
    print(label,path_to_data)
    MTM_retrival_Object.process_measurement(path_to_data, label = label)

MTM 40nm retrieved


# Interactive visualizacion of the data

In [18]:
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
from IPython.display import clear_output
from matplotlib.pyplot import subplots, show, close

In [19]:
eigen_list = [0,1,2]

def plot_MTM(Bandwidth, Eigenstate):
    clear_output(wait=True)
    f, ax = subplots(1, 2, figsize=(15, 5))
    ax[0].imshow(complexToRGB(MTM_retrival_Object.MTM[Bandwidth][Eigenstate]))
    eig = r'$\varphi$'
    ax[0].set_title(f'MTM - $\Delta_\lambda$ {Bandwidth} - ' + eig + f' {Eigenstate}')
    ax[0].set_xlabel('modes out')
    ax[0].set_ylabel('modes in')
    MTM_retrival_Object.showMTMSVD_interact(Bandwidth, ax[1])
    show()
    close(f)

In [20]:
interact(plot_MTM, Bandwidth = labels, Eigenstate = eigen_list) #Interact with the processed data

interactive(children=(Dropdown(description='Bandwidth', options=('0nm', '5nm', '10nm', '20nm', '30nm', '40nm')…

<function __main__.plot_MTM(Bandwidth, Eigenstate)>